# exp001 - LLaMA 3.2 Stage 1 Baseline v2.1

**实验目标**：测量 LLaMA 3.2 系列模型（1B / 3B / 11B-Vision）在 Google Colab GPU 上的推理性能基线。

**测量指标**：TTFT（首 token 时延）/ TPOT（每 token 时延）/ KV Cache 大小 / 峰値显存

**版本**：v2.1 -- 支持 text_only 和 single_image 两种输入模式，forward hook 分项计时视觉编码。

## Section 0: Environment & Dependencies

In [ ]:
# Install required dependencies
# transformers: model loading and inference
# accelerate: device_map='auto' for GPU placement
# huggingface_hub: model download and auth
# ipywidgets: interactive configuration UI
# psutil: system memory monitoring
# pillow: image processing
# datasets: quality evaluation datasets
!pip install -q transformers accelerate huggingface_hub \
             ipywidgets psutil pillow datasets

In [ ]:
# ================================================================
# Path constants -- all paths defined here, not hardcoded elsewhere
# ================================================================
import sys
from pathlib import Path

# Google Drive project root (after mounting)
DRIVE_ROOT      = "/content/drive/MyDrive/EdgeLLM-Systems"

# Model weights cache (persisted on Drive to avoid re-downloading)
MODEL_CACHE_DIR = f"{DRIVE_ROOT}/models/llama32_models"

# Dataset directory (profiling_suite and quality_suite)
DATASET_DIR     = f"{DRIVE_ROOT}/dataset"

# Experiment results output directory
RESULTS_DIR     = f"{DRIVE_ROOT}/results/exp001"

# Project module path (edge_llm_systems added to sys.path after Drive mount)
PROJECT_SRC     = f"{DRIVE_ROOT}"
if PROJECT_SRC not in sys.path:
    sys.path.insert(0, PROJECT_SRC)

print(f"DRIVE_ROOT:      {DRIVE_ROOT}")
print(f"MODEL_CACHE_DIR: {MODEL_CACHE_DIR}")
print(f"DATASET_DIR:     {DATASET_DIR}")
print(f"RESULTS_DIR:     {RESULTS_DIR}")

## Section 1: Mount Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path

# Mount Google Drive -- triggers auth dialog
drive.mount("/content/drive")

# Verify project root exists, create if needed
drive_root_path = Path(DRIVE_ROOT)
if not drive_root_path.exists():
    drive_root_path.mkdir(parents=True, exist_ok=True)
    print(f"[Drive] Created: {DRIVE_ROOT}")
else:
    print(f"[Drive] Exists: {DRIVE_ROOT}")

# 只确保 results/exp001/ 基础目录存在
# raw_runs/ 和 group_summary/ 在 Section 6 按模型名动态创建，不在这里预建
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

print("[Drive] Mount complete, directory structure ready")

## Section 2: HuggingFace Authentication

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# Read HF Token from Colab Secrets (set via key icon in left sidebar)
# LLaMA 3.2 is gated -- accept terms on HuggingFace website first
HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)
print("[HuggingFace] Authentication complete")

## Section 3: Experiment Configuration (Interactive UI)

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# ── 模型选择（exp001：text-only 1B / 3B）────────────────────────────────────
# exp002 会增加 LLaMA-3.2-11B-Vision-Instruct
w_model = widgets.Dropdown(
    options=["LLaMA-3.2-1B", "LLaMA-3.2-3B"],
    value="LLaMA-3.2-1B",
    description="Model:",
    style={"description_width": "initial"},
)

# ── 输入模式（接口预留，exp002 实现 single_image / multi_images）────────────
w_input_mode = widgets.Dropdown(
    options=[
        ("text_only  ✅ exp001",     "text_only"),
        ("single_image  🔒 exp002",  "single_image"),
        ("multi_images  🔒 exp002",  "multi_images"),
    ],
    value="text_only",
    description="Input Mode:",
    style={"description_width": "initial"},
)

# ── 测试开关（两者独立，互不依赖）────────────────────────────────────────────
w_run_perf = widgets.Checkbox(value=True,  description="🚀 Performance Test (Profiling)")
w_run_qual = widgets.Checkbox(value=True,  description="🎯 Quality Evaluation (Benchmarks)")

# ── 性能测试参数 ──────────────────────────────────────────────────────────────
w_prompt_lens = widgets.Text(
    value="64, 128, 256, 512, 1024, 2048",
    description="Prompt Lengths:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)
w_gen_lens = widgets.Text(
    value="32, 64, 128",
    description="Gen Lengths:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)
w_img_resolutions = widgets.Text(
    value="224, 336, 448",
    description="Image Resolutions:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)
w_repeat = widgets.Dropdown(
    options=[1, 3, 5], value=3,
    description="Repeat:",
    style={"description_width": "initial"},
)

# ── 质量评估参数 ──────────────────────────────────────────────────────────────
from edge_llm_systems.quality import BENCHMARK_CONFIGS
w_benchmarks = widgets.SelectMultiple(
    options=list(BENCHMARK_CONFIGS.keys()),
    value=list(BENCHMARK_CONFIGS.keys()),
    description="Benchmarks:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px", height="110px"),
)
w_quality_seed = widgets.IntText(
    value=42,
    description="Seed:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="180px"),
)

# ── 确认按钮 ──────────────────────────────────────────────────────────────────
w_confirm = widgets.Button(
    description="✅ Confirm Config",
    button_style="success",
    layout=widgets.Layout(width="160px"),
)
w_output = widgets.Output()

# ── 响应式显示：切换输入模式时隐藏/显示对应参数 ──────────────────────────────
def _on_mode_change(change):
    is_text = change["new"] == "text_only"
    w_prompt_lens.layout.display     = ""     if is_text else "none"
    w_img_resolutions.layout.display = "none" if is_text else ""
    # vision/multi_images 模式：质量评估暂不支持（exp002）
    if change["new"] != "text_only":
        w_run_qual.value = False
        w_run_qual.disabled = True
        w_benchmarks.disabled = True
    else:
        w_run_qual.disabled = False
        w_benchmarks.disabled = False

w_input_mode.observe(_on_mode_change, names="value")
w_img_resolutions.layout.display = "none"   # 初始隐藏

# ── 全局配置字典（后续 Section 均从 CONFIG 读取）──────────────────────────────
CONFIG = {}

def _on_confirm(b):
    global CONFIG
    with w_output:
        w_output.clear_output()
        def parse_ints(s):
            return [int(x.strip()) for x in s.split(",") if x.strip()]

        CONFIG = {
            "model_key":         w_model.value,
            "input_mode":        w_input_mode.value,
            "run_performance":   w_run_perf.value,
            "run_quality":       w_run_qual.value,
            "prompt_lengths":    parse_ints(w_prompt_lens.value),
            "gen_lengths":       parse_ints(w_gen_lens.value),
            "image_resolutions": parse_ints(w_img_resolutions.value),
            "repeat":            w_repeat.value,
            # 质量评估参数
            "benchmarks":        list(w_benchmarks.value),
            "quality_seed":      w_quality_seed.value,
        }

        mode = CONFIG["input_mode"]
        if mode != "text_only":
            print(f"⚠️  {mode} 模式在 exp002 中实现，当前仅支持 text_only。")
            return

        yn = lambda v: "✅" if v else "⬜"
        print("✅ 配置已确认：")
        print(f"   模型         : {CONFIG['model_key']}")
        print(f"   输入模式     : {CONFIG['input_mode']}")
        print(f"   性能测试     : {yn(CONFIG['run_performance'])}")
        print(f"   质量评估     : {yn(CONFIG['run_quality'])}")
        if CONFIG["input_mode"] == "text_only":
            print(f"   Prompt Lengths : {CONFIG['prompt_lengths']}")
        else:
            print(f"   Image Resolutions: {CONFIG['image_resolutions']}")
        print(f"   Gen Lengths  : {CONFIG['gen_lengths']}")
        print(f"   Repeat       : {CONFIG['repeat']}")
        if CONFIG["run_quality"]:
            print(f"   Benchmarks   : {CONFIG['benchmarks']}")
            print(f"   Quality Seed : {CONFIG['quality_seed']}")

w_confirm.on_click(_on_confirm)

display(widgets.VBox([
    widgets.HTML("<b>── Model & Mode ──</b>"),
    widgets.HBox([w_model, w_input_mode]),
    widgets.HTML("<b>── Test Switches ──</b>"),
    widgets.HBox([w_run_perf, w_run_qual]),
    widgets.HTML("<b>── Performance Config ──</b>"),
    w_prompt_lens,
    w_img_resolutions,
    w_gen_lens,
    w_repeat,
    widgets.HTML("<b>── Quality Config ──</b>"),
    w_benchmarks,
    w_quality_seed,
    w_confirm,
    w_output,
]))

## Section 4: Model Loading

In [ ]:
import torch
from edge_llm_systems.models import (
    check_model_integrity, download_model_if_needed,
    load_text_model, load_vision_model,
    get_model_load_mem_mb, MODEL_REGISTRY,
)
from edge_llm_systems.utils import log

assert CONFIG, "Please run Section 3 and click Confirm first"

model_key  = CONFIG["model_key"]
input_mode = CONFIG["input_mode"]
device     = "cuda" if torch.cuda.is_available() else "cpu"

log(f"[Model Load] Target: {model_key}")
log(f"[Model Load] Mode: {input_mode}")
log(f"[Model Load] Device: {device}")

# Integrity check + conditional download (cached on Drive)
local_model_path = download_model_if_needed(
    model_key=model_key,
    cache_dir=MODEL_CACHE_DIR,
    hf_token=HF_TOKEN,
)

# Load model based on input mode
# NOTE: LLaMA-3.2-11B-Vision FP16 OOM records oom_stage='model_load', NO fallback
if input_mode == "text_only":
    model, tokenizer = load_text_model(local_model_path, HF_TOKEN)
    processor = None
else:  # single_image
    model, processor = load_vision_model(local_model_path, HF_TOKEN)
    tokenizer = processor.tokenizer

# Record baseline GPU memory (model weights only, before any inference)
model_load_mem_mb = get_model_load_mem_mb(device)

cfg = model.config
num_layers   = getattr(cfg, "num_hidden_layers", "N/A")
num_kv_heads = getattr(cfg, "num_key_value_heads",
                        getattr(cfg, "num_attention_heads", "N/A"))

log(f"  Path:     {local_model_path}")
log(f"  Device:   {model.device}")
log(f"  Layers:   {num_layers}")
log(f"  KV Heads: {num_kv_heads}")
log(f"  Base Mem: {model_load_mem_mb:.1f} MB")

## Section 5: Warm-up

In [ ]:
from edge_llm_systems.runners import run_warmup_text, run_warmup_vision
from edge_llm_systems.utils import log

log("[Warmup] Starting (results NOT written to CSV)...")

if input_mode == "text_only":
    # Short sequences to warm up CUDA kernels and cuDNN autotuning
    run_warmup_text(model=model, tokenizer=tokenizer, device=device)
else:
    # Vision warmup: text first, then image
    from PIL import Image as PILImage
    # Solid-color image -- ensures image preprocessing path is also warmed up
    warmup_image = PILImage.new("RGB", (224, 224), color=(128, 128, 128))
    run_warmup_vision(model=model, processor=processor, device=device,
                      warmup_image=warmup_image)

log("[Warmup] Complete")

## Section 6: Performance Benchmarking

In [ ]:
from pathlib import Path
from edge_llm_systems.runners import run_benchmark_text, run_benchmark_vision
from edge_llm_systems.utils import (
    log, generate_run_id, build_timestamp_filename, model_hash,
    collect_environment_info, collect_model_info, collect_run_config, save_json,
)
from edge_llm_systems.models import MODEL_REGISTRY

RUN_PERFORMANCE = CONFIG.get("run_performance", True)

if not RUN_PERFORMANCE:
    print("[Profiling] Not enabled, skipping")
else:
    run_id   = generate_run_id()
    model_id = MODEL_REGISTRY[model_key]

    # model_slug：HuggingFace repo 最后一段，作为结果子目录名
    # 例：meta-llama/Llama-3.2-1B-Instruct → Llama-3.2-1B-Instruct
    model_slug = model_id.split("/")[-1]

    # ── 结果目录（按模型名隔离）──────────────────────────────────────────────
    # MyDrive/EdgeLLM-Systems/results/exp001/{model_slug}/
    # ├── perf_raw_{ts}.csv          性能原始数据
    # ├── perf_summary_{ts}.csv      性能组均值
    # ├── qual_raw_{bm}_{ts}.csv     质量 per-sample（Section 7 写入）
    # ├── qual_summary_{ts}.json     质量汇总（Section 7 写入）
    # └── exp_info/
    #     ├── environment/
    #     ├── model/
    #     └── run_config/
    model_result_dir = Path(f"{RESULTS_DIR}/{model_slug}")
    exp_info_env_dir = model_result_dir / "exp_info" / "environment"
    exp_info_mdl_dir = model_result_dir / "exp_info" / "model"
    exp_info_cfg_dir = model_result_dir / "exp_info" / "run_config"

    for d in [model_result_dir, exp_info_env_dir, exp_info_mdl_dir, exp_info_cfg_dir]:
        d.mkdir(parents=True, exist_ok=True)

    # 带时间戳的文件名，防止多次运行覆盖历史数据
    raw_csv_path     = str(model_result_dir / build_timestamp_filename("perf_raw",     "csv"))
    summary_csv_path = str(model_result_dir / build_timestamp_filename("perf_summary", "csv"))

    # ── 保存 exp_info JSON 三件套 ─────────────────────────────────────────────
    env_info = collect_environment_info(
        run_id=run_id, model_id=model_id, model_key=model_key,
        raw_csv_path=raw_csv_path, summary_csv_path=summary_csv_path,
    )
    env_path = exp_info_env_dir / f"{run_id}_environment.json"
    save_json(env_path, env_info)
    log(f"[Exp Info] environment → {env_path.name}")

    model_info_path = exp_info_mdl_dir / f"{model_slug}_model_info.json"
    if not model_info_path.exists():
        model_info = collect_model_info(
            model=model, model_id=model_id, model_key=model_key,
            local_model_path=local_model_path,
        )
        save_json(model_info_path, model_info)
        log(f"[Exp Info] model_info → {model_info_path.name}")
    else:
        log(f"[Exp Info] model_info already exists, skipped")

    run_cfg = collect_run_config(
        run_id=run_id, model_id=model_id, model_key=model_key,
        config=CONFIG, raw_csv_path=raw_csv_path, summary_csv_path=summary_csv_path,
    )
    run_cfg_path = exp_info_cfg_dir / f"{run_id}_run_config.json"
    save_json(run_cfg_path, run_cfg)
    log(f"[Exp Info] run_config → {run_cfg_path.name}")

    # ── 每行 CSV 附加的元数据 ──────────────────────────────────────────────────
    run_meta = {
        "run_id":     run_id,
        "model_id":   model_id,
        "model_hash": model_hash(model_id),
        "input_mode": input_mode,
        "test_type":  "profiling",
    }

    log(f"[Profiling] run_id      : {run_id}")
    log(f"[Profiling] model_slug  : {model_slug}")
    log(f"[Profiling] perf_raw    : {Path(raw_csv_path).name}")
    log(f"[Profiling] perf_summary: {Path(summary_csv_path).name}")

    if input_mode == "text_only":
        run_benchmark_text(
            model=model, tokenizer=tokenizer, device=device,
            prompt_lengths=CONFIG["prompt_lengths"],
            gen_lengths=CONFIG["gen_lengths"],
            repeat=CONFIG["repeat"],
            model_load_mem_mb=model_load_mem_mb,
            raw_csv_path=raw_csv_path,
            summary_csv_path=summary_csv_path,
            run_meta=run_meta,
        )
    else:
        # single_image / multi_images — exp002
        print(f"[Profiling] {input_mode} 模式在 exp002 中实现")

## Section 7: Quality Evaluation

Evaluates model accuracy on standard benchmarks.

**Text-only benchmarks (5 total):**
- **MMLU-Pro mini** (70 samples): College-level MCQ, A/B/C/D choice
- **GSM8K mini** (50 samples): Math word problems, extract final number
- **HellaSwag mini** (50 samples): Commonsense reasoning, sentence completion
- **WinoGrande mini** (50 samples): Pronoun resolution, binary choice
- **TruthfulQA-MC** (50 samples): Factual MCQ

**Single-image benchmarks (5 total):**
- **VQAv2 mini** (50 samples): General VQA, soft exact match
- **MMBench mini** (50 samples): Multi-dimensional vision MCQ
- **MathVista mini** (30 samples): Visual math reasoning, MCQ + numeric
- **TextVQA mini** (30 samples): OCR-based VQA, soft match
- **DocVQA mini** (30 samples): Document VQA, ANLS score

In [ ]:
# Section 7 辅助：从 quality 模块导入，不在 notebook 中内联实现
# 此 cell 只做导入验证，实际逻辑全部在 edge_llm_systems/quality.py

from edge_llm_systems.quality import (
    BENCHMARK_CONFIGS,
    VISION_BENCHMARK_CONFIGS,
    run_text_quality_suite,
    run_vision_quality_suite,   # 预留接口，exp002 实现
)

print("[Quality] 模块导入成功")
print(f"  文本基准 ({len(BENCHMARK_CONFIGS)}): {list(BENCHMARK_CONFIGS.keys())}")
print(f"  视觉基准 ({len(VISION_BENCHMARK_CONFIGS)}，exp002): {list(VISION_BENCHMARK_CONFIGS.keys())}")

In [ ]:
from pathlib import Path
from edge_llm_systems.quality import run_text_quality_suite
from edge_llm_systems.utils import log

RUN_QUALITY = CONFIG.get("run_quality", False)
input_mode  = CONFIG.get("input_mode", "text_only")

if not RUN_QUALITY:
    print("[Quality] 未启用（RUN_QUALITY=False），跳过")

elif input_mode != "text_only":
    # single_image / multi_images 模式的质量评估在 exp002 中实现
    print(f"[Quality] {input_mode} 模式的质量评估在 exp002 中实现，当前跳过")

else:
    # ── 确认 run_id 和 model_slug 已在 Section 6 定义 ─────────────────────────
    # 若 Section 6 (性能测试) 被跳过，则在此处补充生成
    if "run_id" not in dir() or not run_id:
        from edge_llm_systems.utils import generate_run_id
        from edge_llm_systems.models import MODEL_REGISTRY
        run_id   = generate_run_id()
        model_id = MODEL_REGISTRY[CONFIG["model_key"]]
        model_slug = model_id.split("/")[-1]
        log(f"[Quality] Section 6 未运行，生成新 run_id: {run_id}")

    model_result_dir = Path(f"{RESULTS_DIR}/{model_slug}")

    log(f"[Quality] 开始文本质量评估")
    log(f"[Quality] run_id   : {run_id}")
    log(f"[Quality] model    : {model_id}")
    log(f"[Quality] benchmarks: {CONFIG['benchmarks']}")
    log(f"[Quality] seed     : {CONFIG['quality_seed']}")

    # ── 运行质量评估套件 ──────────────────────────────────────────────────────
    # 结果写入：
    #   {RESULTS_DIR}/{model_slug}/quality_raw/{run_id}_{benchmark}_raw.csv
    #   {RESULTS_DIR}/{model_slug}/quality_summary/{run_id}_quality_summary.json
    quality_results = run_text_quality_suite(
        model=model,
        tokenizer=tokenizer,
        device=device,
        dataset_dir=DATASET_DIR,
        model_result_dir=str(model_result_dir),
        run_id=run_id,
        model_id=model_id,
        benchmarks=CONFIG["benchmarks"],
        seed=CONFIG["quality_seed"],
    )

    log("[Quality] ✅ 文本质量评估完成")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Section 7b：视觉质量评估 — 🔒 exp002 预留
# ══════════════════════════════════════════════════════════════════════════════
#
# exp001 范围：text-only（LLaMA-3.2-1B / 3B）
# exp002 范围：single_image / multi_images（LLaMA-3.2-11B-Vision）
#
# 接口已在 edge_llm_systems/quality.py 中预留：
#   run_vision_quality_suite(model, processor, device, dataset_dir,
#                            model_result_dir, run_id, model_id,
#                            benchmarks, seed)
#
# 支持的视觉基准（exp002 实现）：
#   - vqav2_mini     (50 samples)：通用视觉问答，soft exact match
#   - mmbench_mini   (50 samples)：多维视觉 MCQ
#   - mathvista_mini (30 samples)：视觉数学推理
#   - textvqa_mini   (30 samples)：OCR 密集型 VQA
#   - docvqa_mini    (30 samples)：文档理解，ANLS 评分
# ══════════════════════════════════════════════════════════════════════════════

print("[Quality] 视觉质量评估接口已预留，将在 exp002 中实现。")
print("  调用方式（exp002）：")
print("    from edge_llm_systems.quality import run_vision_quality_suite")
print("    results = run_vision_quality_suite(model, processor, device, ...)")

## Section 8: Results Summary

In [ ]:
import json
import pandas as pd
from pathlib import Path
from edge_llm_systems.utils import log, read_csv_utf8sig
from edge_llm_systems.models import MODEL_REGISTRY

log("[Results] ══════════════ Experiment Summary ══════════════")

# model_slug 在 Section 6/7 中定义；跳过两者时从 CONFIG 重新推导
if "model_slug" not in dir() or not model_slug:
    model_slug = MODEL_REGISTRY.get(CONFIG.get("model_key", ""), "").split("/")[-1]
    if not model_slug:
        print("[Results] ⚠️  无法推导 model_slug，请先运行 Section 6 或 7")
        raise SystemExit

model_result_dir = Path(f"{RESULTS_DIR}/{model_slug}")

# ── 列出所有产出文件（平铺结构）────────────────────────────────────────────
print(f"\n📁 results/exp001/{model_slug}/")

all_files = sorted(f for f in model_result_dir.glob("*") if f.is_file())
exp_info_files = sorted(model_result_dir.glob("exp_info/**/*"))

if all_files:
    for f in all_files:
        size_kb = f.stat().st_size / 1024
        icon = "📊" if f.suffix == ".csv" else "📋" if f.suffix == ".json" else "📄"
        print(f"  {icon} {f.name}  ({size_kb:.1f} KB)")
if exp_info_files:
    print(f"  📁 exp_info/")
    for f in exp_info_files:
        if f.is_file():
            size_kb = f.stat().st_size / 1024
            rel = f.relative_to(model_result_dir)
            print(f"    📋 {rel}  ({size_kb:.1f} KB)")

# ── 性能测试摘要表 ──────────────────────────────────────────────────────────
print("\n" + "─" * 64)
print("🚀 Performance Summary  (perf_summary_*.csv — group means)")
print("─" * 64)
perf_files = sorted(model_result_dir.glob("perf_summary_*.csv"))
if perf_files:
    latest = perf_files[-1]
    rows = read_csv_utf8sig(str(latest))
    if rows:
        df = pd.DataFrame(rows)
        df_mean = df[df["run_index"].astype(str) == "0"].copy()
        display_cols = [c for c in [
            "prompt_len", "gen_len",
            "ttft_ms", "tpot_ms", "tokens_s",
            "peak_mem_mb", "kv_pkv_final_mb", "status",
        ] if c in df_mean.columns]
        print(f"  Source: {latest.name}")
        print(df_mean[display_cols].to_string(index=False))
    else:
        print("  (empty)")
else:
    print("  No perf_summary CSV found — run Section 6 first")

# ── 质量评估摘要表 ──────────────────────────────────────────────────────────
print("\n" + "─" * 64)
print("🎯 Quality Summary  (qual_summary_*.json)")
print("─" * 64)
qual_files = sorted(model_result_dir.glob("qual_summary_*.json"))
if qual_files:
    latest_q = qual_files[-1]
    with open(latest_q, encoding="utf-8") as f:
        qs = json.load(f)
    print(f"  Source: {latest_q.name}")
    print(f"  Model : {qs.get('model_id', 'N/A')}")
    print(f"  Seed  : {qs.get('seed', 'N/A')}")
    print(f"  Mean Accuracy: {qs.get('mean_accuracy', 0):.1f}%\n")
    header = f"  {'Benchmark':<22} {'Accuracy':>9}  {'Correct/Total':>14}  {'Ans%':>6}"
    print(header)
    print("  " + "─" * (len(header) - 2))
    for r in qs.get("benchmarks", []):
        frac = f"{r['num_correct']}/{r['num_samples']}"
        print(f"  {r['benchmark']:<22} {r['accuracy']:>8.1f}%"
              f"  {frac:>14}  {r['answer_rate']:>5.0f}%")
else:
    print("  No qual_summary JSON found — run Section 7 first")

log("[Results] ✅ Summary complete")